In [ ]:
import os, subprocess
repo_dir = "slot-imputation"
if os.path.isdir(repo_dir):
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "https://github.com/jdbohrman/slot-imputation.git"], check=True)


In [ ]:
%pip install -q -e slot-imputation seaborn matplotlib


In [ ]:
import json, math, time
import torch
import torch.nn.functional as F

from slot_impute.transformer import TransformerConfig, MiniGPT2
from slot_impute.data import load_wikitext_batches, wikitext_batch_iterator
from slot_impute.train_transformer import train_model
from slot_impute.impute_mlp import impute_mlp_to_target
from slot_impute.validate_mlp import compute_ppl, weight_distance, mlp_ablation_gap, convergence_head_start, compute_cost_savings, compute_dollar_savings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Phase 1 — Quick Test

Tiny model proves the upsampling works before committing to a long run.

In [ ]:
cfg = TransformerConfig(d_model=128, d_ff=128, n_heads=4, n_layers=2, max_seq_len=128, vocab_size=50257)
model = MiniGPT2(cfg).to(DEVICE)
print(f"Model: d_model={cfg.d_model} d_ff={cfg.d_ff} params={model.num_parameters():,}")

batches = load_wikitext_batches(batch_size=16, seq_len=128, split="train", subset_tokens=32768, device=DEVICE)
print(f"Data: {len(batches)} batches")

t0 = time.time()
losses = train_model(model, batches, DEVICE, steps=300, lr=6e-4, seed=42, warmup_steps=30, grad_clip=1.0)
print(f"Training done ({time.time() - t0:.0f}s). Final loss: {sum(losses[-30:])/30:.4f}")

In [ ]:
val = load_wikitext_batches(batch_size=16, seq_len=128, split="validation", device=DEVICE)

imputed = impute_mlp_to_target(model, 256, val, num_act_batches=5)
print(f"Imputed d_ff={imputed.config.d_ff}, params={imputed.num_parameters():,}")

In [ ]:
@torch.no_grad()
def ppl(m, batches, n=5):
    m.eval(); m.to(DEVICE)
    total, cnt = 0.0, 0
    for b in batches[:n]:
        x = b["input_ids"].to(DEVICE); y = b["labels"].to(DEVICE)
        loss = F.cross_entropy(m(x).contiguous().view(-1, m.config.vocab_size), y.contiguous().view(-1))
        total += loss.item(); cnt += 1
    return math.exp(min(total / max(cnt, 1), 20))

ppl_src  = ppl(model,   val)
ppl_imp  = ppl(imputed, val)
ppl_rand = ppl(MiniGPT2(imputed.config), val)

print(f"d_ff=128 source:  {ppl_src:.1f}")
print(f"d_ff=256 imputed: {ppl_imp:.1f}")
print(f"d_ff=256 random:  {ppl_rand:.1f}")
print()
if ppl_imp < ppl_src:  # imputed beats source model despite no training at d_ff=256
    print(f"PASSED — imputed PPL is {ppl_src - ppl_imp:.0f} below source ({ppl_imp/ppl_src:.2f}x)")
else:
    print(f"WEAK — imputed PPL {ppl_imp - ppl_src:.0f} above source (imputation didn't transfer)")

In [ ]:
# Projected cost savings at GPT-2 scale (d_model=768, n_layers=12)
ds_proj = compute_dollar_savings(128, 256, 768, 12, steps=20000, gpu_cost_per_hour=2.0)
print(f"MLP FLOPs ratio:  {ds_proj['mlp_flops_ratio']}x")
print(f"Block FLOPs ratio: {ds_proj['block_flops_ratio']}x  (attention unchanged, MLP 2x)")
print(f"Compute savings:   {ds_proj['compute_savings_pct']}%")
print(f"")
print(f"To train this 12-layer model on {ds_proj['gpu']} for {ds_proj['steps']:,} steps:")
print(f"  Slim ({ds_proj['d_ff_small']}): {ds_proj['training_time_slim']}")
print(f"  Full ({ds_proj['d_ff_large']}): {ds_proj['training_time_full']}")


## Phase 2 — Full Experiment

GPT-2 scale: d_model=768, n_layers=12, d_ff=2048→4096. ~6-8h on T4 GPU.

In [ ]:
cfg_src = TransformerConfig(d_model=768, d_ff=2048, n_heads=12, n_layers=12, max_seq_len=1024)
cfg_gt  = TransformerConfig(d_model=768, d_ff=4096, n_heads=12, n_layers=12, max_seq_len=1024)
model_src = MiniGPT2(cfg_src).to(DEVICE)
model_gt  = MiniGPT2(cfg_gt).to(DEVICE)
print(f"Source (d_ff=2048):      {model_src.num_parameters():,} params")
print(f"Ground truth (d_ff=4096): {model_gt.num_parameters():,} params")

In [ ]:
batches = load_wikitext_batches(batch_size=8, seq_len=1024, split="train", device=DEVICE)
print(f"Train data: {len(batches)} batches")

print("Training source (d_ff=2048)...")
t_src_start = time.time()
train_model(model_src, batches, DEVICE, steps=20000, lr=6e-4, seed=42, warmup_steps=1000, grad_clip=1.0, save_dir="checkpoints_dff", checkpoint_tag="source")
t_src_sec = time.time() - t_src_start
ms_per_step_src = t_src_sec * 1000 / 20000
print(f"  Source training time: {t_src_sec:.0f}s ({ms_per_step_src:.1f} ms/step)")

print("Training ground truth (d_ff=4096)...")
t_gt_start = time.time()
train_model(model_gt, batches, DEVICE, steps=20000, lr=6e-4, seed=42, warmup_steps=1000, grad_clip=1.0, save_dir="checkpoints_dff", checkpoint_tag="ground_truth")
t_gt_sec = time.time() - t_gt_start
ms_per_step_gt = t_gt_sec * 1000 / 20000
print(f"  Ground truth training time: {t_gt_sec:.0f}s ({ms_per_step_gt:.1f} ms/step)")


In [ ]:
val = load_wikitext_batches(batch_size=8, seq_len=1024, split="validation", device=DEVICE)
imputed = impute_mlp_to_target(model_src, 4096, val, num_act_batches=10)

ppl_src  = compute_ppl(model_src, val, DEVICE, 20)
ppl_imp  = compute_ppl(imputed,   val, DEVICE, 20)
ppl_gt   = compute_ppl(model_gt,  val, DEVICE, 20)
ppl_rand = ppl(MiniGPT2(imputed.config), val, 20)

wd = weight_distance(imputed, model_gt)
abl = mlp_ablation_gap(model_src, val, DEVICE, 10)
conv = convergence_head_start(imputed, val, DEVICE, 200)
ds = compute_dollar_savings(2048, 4096, 768, 12, steps=20000,
    ms_per_step_small=ms_per_step_src, ms_per_step_large=ms_per_step_gt, gpu_cost_per_hour=2.0)

print(f"PPL — src:{ppl_src:.1f} imp:{ppl_imp:.1f} gt:{ppl_gt:.1f} rand:{ppl_rand:.1f}")
print(f"Weight — W1 mse:{wd['W1_mse']:.4f} cos:{wd['W1_cosine']:.3f} W2 mse:{wd['W2_mse']:.4f} cos:{wd['W2_cosine']:.3f}")
print(f"MLP gap — with:{abl['ppl_with_mlp']:.1f} without:{abl['ppl_without_mlp']:.1f} gap:{abl['mlp_gap']:.1f}")
print(f"Convergence — speedup:{conv['speedup_ratio']:.3f} ({conv['speedup_steps']}/200)")
print(f"Cost \u2014 {ds['compute_savings_pct']}% compute savings ({ds['block_flops_ratio']}x FLOPs ratio)")

print()
print(f"Dollar analysis ({ds['gpu']}):")
print(f"  Train slim ({ds['d_ff_small']}): {ds['training_time_slim']} \u2192 {ds['cost_train_slim']}")
print(f"  Train full ({ds['d_ff_large']}): {ds['training_time_full']} \u2192 {ds['cost_train_full']}")
print(f"  Imputation: {ds['cost_imputation']}")
print(f"  Savings: {ds['cost_savings']} ({ds['compute_savings_pct']}%)")
json.dump({
    "ppl": {"source": ppl_src, "imputed": ppl_imp, "ground_truth": ppl_gt, "random": ppl_rand},
    "weight_distance": wd, "ablation": abl, "convergence": conv, "cost": ds,
}, open("dff_results.json", "w"), indent=2, default=str)
print("Saved dff_results.json")

## Phase 3 — Cost Scaling Analysis

How dollar savings scale with model size, MLP upsample ratio, and GPU type.
Uses FLOPs-based training cost estimates (forward+backward ≈ 6× params per token).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

# Parameter grid
DM       = [128, 256, 512, 768, 1024, 2048, 4096]
NL       = [2, 4, 8, 12, 24, 48]
RATIOS   = [2, 4, 8]
GPUS     = {"T4": (0.35, 65), "A100": (2.0, 312), "H100": (4.0, 990)}
STEPS    = 20_000; BATCH = 8; SEQ = 1024; VOCAB = 50257; UTIL = 0.5; FF = 4

def est_cost(dm, dff, nl, cost_hr, tflops):
    total_params = VOCAB*dm + 1024*dm + 4*dm*dm*nl + 2*dm*dff*nl
    flops = 6 * total_params * BATCH * SEQ * STEPS
    hr = flops / (tflops * 1e12 * UTIL) / 3600
    return hr * cost_hr, hr, total_params / 1e6

# Build dataframe
rows = []
for dm in DM:
    for nl in NL:
        dff_s = FF * dm
        for r in RATIOS:
            dff_l = r * dff_s
            mlp_s, mlp_l = 2*dm*dff_s*nl, 2*dm*dff_l*nl
            attn = 4*dm*dm*nl
            block_r = (attn + mlp_l) / (attn + mlp_s)
            for gpu, (ch, tf) in GPUS.items():
                c_full, _, p_full = est_cost(dm, dff_l, nl, ch, tf)
                c_slim, _, p_slim = est_cost(dm, dff_s, nl, ch, tf)
                savings = max(c_full - c_slim, 0.01)
                rows.append(dict(d_model=dm, n_layers=nl, ratio=r, gpu=gpu,
                    params_slim_M=round(p_slim), params_full_M=round(p_full),
                    block_ratio=round(block_r,2), savings=round(savings,2),
                    log10_savings=np.log10(savings)))
df = pd.DataFrame(rows)

# --- Plot 1: Heatmap of log savings by upsample ratio x model scale ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, (gpu, _) in zip(axes, GPUS.items()):
    sub = df[df.gpu == gpu].pivot_table(index="ratio", columns=["d_model","n_layers"],
        values="log10_savings", aggfunc="first")
    sub.columns = [f"d={c[0]}\u00d7{c[1]}L" for c in sub.columns]
    sns.heatmap(sub, annot=True, fmt=".1f", cmap="rocket_r", ax=ax,
                cbar_kws={"label": "log₁₀($ Savings)"}, linewidths=0.5, vmin=0, vmax=4)
    ax.set_title(f"{gpu} @ ${GPUS[gpu][0]:.2f}/hr", fontweight="bold")
    ax.set_ylabel("MLP Upsample Ratio"); ax.set_xlabel("Model Scale (d_model × n_layers)")
fig.suptitle("Log-Scale Training Cost Savings", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# --- Plot 2: Savings curves: log(params) vs log(savings) ---
fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette("Set2", len(GPUS))
for (gpu, _), color in zip(GPUS.items(), palette):
    sub = df[df.gpu == gpu]
    for r, ls in zip(RATIOS, ["-", "--", ":"]):
        sub_r = sub[sub.ratio == r].sort_values("params_slim_M")
        ax.plot(sub_r.params_slim_M, sub_r.log10_savings, color=color, linestyle=ls,
                marker="o", markersize=3, alpha=0.7,
                label=f"{gpu}, {r}×" if ls == "-" else f"{gpu}, {r}×" if gpu == list(GPUS)[0] else f"{r}×")
ax.set_xscale("log")
ax.set_xlabel("Slim Model Params (M)", fontsize=12)
ax.set_ylabel("log₁₀($ Savings)", fontsize=12)
ax.set_title("Log Savings vs Model Scale", fontweight="bold", fontsize=13)
ax.legend(ncol=3, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

# --- Plot 3: % savings histogram showing distribution ---
fig, ax = plt.subplots(figsize=(10, 4))
df_pct = df.drop_duplicates(subset=["d_model","n_layers","ratio"]).copy()
df_pct["savings_pct"] = (1 - 1/df_pct.block_ratio) * 100
sns.boxplot(data=df_pct, x="ratio", y="savings_pct", hue="ratio",
            palette="flare", legend=False, ax=ax)
sns.stripplot(data=df_pct, x="ratio", y="savings_pct", color="black", size=2, alpha=0.3, ax=ax)
ax.set_xlabel("MLP Upsample Ratio", fontsize=12)
ax.set_ylabel("Compute Savings (%)", fontsize=12)
ax.set_title("FLOPs Savings Distribution by Upsample Ratio", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

# --- Summary table ---
summary = df.groupby(["gpu","ratio"]).agg(
    min_save=("savings","min"), med_save=("savings","median"),
    max_save=("savings","max"), count=("savings","count")
).round(1)
print("\nDollar savings range by GPU & upsample ratio (20K steps):")
print(summary.to_string())
